In [ ]:
# !pip install "numpy<2" faiss-cpu --user
# !pip install --upgrade numexpr bottleneck --user

!pip install chromadb --user -qqq



In [1]:
import json

def load_content(filepath):
    with open(filepath, "r") as f:
        content = f.read()
        return content

def load_json(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)
        return data

In [2]:
import os
import pandas as pd

import chromadb

dut_text = load_content("assets/duts/003.md")

requerimentos_folder = "assets/medical_requirements"
requerimentos_filepaths = sorted([f"{requerimentos_folder}/{x}" for x in os.listdir(requerimentos_folder) if x[:5] == "case_" and x[-4:] == ".txt"])
requerimentos = [{"id": x.split("/")[-1], "dut_number": 3, "dut_text": dut_text, "requerimento_text": load_content(x)} for x in requerimentos_filepaths]


# Gerar Resumos

In [3]:
import asyncio
import os
import json
from dataclasses import dataclass, asdict
from datetime import datetime
from tqdm.asyncio import tqdm

import faiss
import openai
import numpy as np
from openai import OpenAI




CRIAR_RESUMOS = True
annotations_filepath = "assets/annotations.json"
vector_db_content_filepath = "assets/vector_database/data.json"
# -------------------------------------------------------------------
# Prompts
# -------------------------------------------------------------------

SYSTEM_PROMPT_RESUMO = """
Você é um especialista em auditoria médica de planos de saúde.

Sua função é ler o texto de uma DUT e o documento de solicitação médica
e produzir um resumo clínico conciso, objetivo e padronizado.

O resumo deve capturar:
- Qual a DUT acionada
- O procedimento solicitado
- Os critérios da DUT que estão sendo invocados
- Os dados clínicos relevantes do paciente (idade, sexo, diagnóstico, sintomas)
- Os exames e documentos apresentados
- O resultado de cálculos clínicos mencionados (escores, probabilidades)
- As lacunas documentais, se houver

O resumo deve ser denso em informação clínica e curto — entre 5 e 10 linhas.
Não emita julgamento sobre conformidade. Apenas descreva o caso.
"""

USER_PROMPT_RESUMO = """
## DUT

{dut_text}

---

## REQUERIMENTO MÉDICO

{requerimento_text}

---

Produza o resumo clínico do caso acima. Apenas o texto do resumo, sem títulos,
sem marcadores, sem formatação.
"""



# -------------------------------------------------------------------
# Geração do resumo clínico
# -------------------------------------------------------------------
async def gerar_resumo_clinico(client, dut_text: str, requerimento_text: str, model="gpt-5-mini", temperature: float = 1) -> str:
    response = await client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_RESUMO},
            {"role": "user",   "content": USER_PROMPT_RESUMO.format(
                dut_text=dut_text,
                requerimento_text=requerimento_text,
            )},
        ],
    )
    return response.choices[0].message.content.strip()

if CRIAR_RESUMOS:
    client = openai.AsyncOpenAI()
    async def processar_resumos(client):
        tasks = [
            gerar_resumo_clinico(client, r["dut_text"], r["requerimento_text"])
            for r in requerimentos
        ]
        resultados = await tqdm.gather(*tasks, desc="Gerando resumos")
        for index, resumo in enumerate(resultados):
            requerimentos[index]["resumo_clinico"] = resumo

    await processar_resumos(client)
    pd.DataFrame(requerimentos).merge(pd.read_json(annotations_filepath), on="id").to_json(vector_db_content_filepath, orient="records")

Gerando resumos:   0%|          | 0/10 [00:00<?, ?it/s]

Gerando resumos:  10%|█         | 1/10 [00:11<01:40, 11.19s/it]

Gerando resumos:  20%|██        | 2/10 [00:11<00:39,  4.94s/it]

Gerando resumos:  30%|███       | 3/10 [00:12<00:19,  2.84s/it]

Gerando resumos:  40%|████      | 4/10 [00:12<00:10,  1.78s/it]

...

Gerando resumos:  60%|██████    | 6/10 [00:14<00:05,  1.47s/it]

Gerando resumos:  80%|████████  | 8/10 [00:15<00:02,  1.05s/it]

Gerando resumos:  90%|█████████ | 9/10 [00:17<00:01,  1.23s/it]

Gerando resumos: 100%|██████████| 10/10 [00:17<00:00,  1.03s/it]

Gerando resumos: 100%|██████████| 10/10 [00:17<00:00,  1.79s/it]

# Gerar Vector Database

In [4]:
import chromadb

@dataclass
class Documento:
    caso_id: str
    resumo_clinico: str
    metadados: dict
    embedding: list[float] | None = None

client = openai.AsyncOpenAI()

# -------------------------------------------------------------------
# Embedding
# -------------------------------------------------------------------
async def gerar_embedding(client, texto: str) -> list[float]:
    response = await client.embeddings.create(
        model="text-embedding-3-small",
        input=texto,
    )
    return response.data[0].embedding

# -------------------------------------------------------------------
# ChromaStore
# -------------------------------------------------------------------
class ChromaStore:
    COLLECTION = "casos"

    def __init__(self, path: str = "store"):
        self._client = chromadb.PersistentClient(path=path)
        self._col = self._client.get_or_create_collection(
            name=self.COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )

    def adicionar(self, doc: Documento) -> None:
        if doc.embedding is None:
            raise ValueError(f"Documento '{doc.caso_id}' sem embedding.")
        if self._col.get(ids=[doc.caso_id])["ids"]:
            raise ValueError(f"Documento '{doc.caso_id}' já indexado.")
        self._col.add(
            ids=[doc.caso_id],
            embeddings=[doc.embedding],
            documents=[doc.resumo_clinico],
            metadatas=[doc.metadados],
        )

    async def buscar(self, query: str, top_k: int = 3) -> list[tuple[Documento, float]]:
        embedding_query = await gerar_embedding(query)
        results = self._col.query(
            query_embeddings=[embedding_query],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
        return [
            (
                Documento(
                    caso_id=caso_id,
                    resumo_clinico=resumo,
                    metadados=meta,
                ),
                1 - distance,  # chroma retorna distância cosine; converte para similaridade
            )
            for caso_id, resumo, meta, distance in zip(
                results["ids"][0],
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

# -------------------------------------------------------------------
# Indexação
# -------------------------------------------------------------------
store = ChromaStore("assets/vector_database/db")
vector_db_content_filepath = "assets/vector_database/data.json"

def format_case(record):
    return {
        "caso_id": record["id"],
        "dut_id": record["dut_number"],
        "dut_text": record["dut_text"],
        "requerimento_text": record["requerimento_text"],
        "informacoes_adicionais": "",
        "resumo_clinico": record["resumo_clinico"],
        "veredicto": record["veredicto"],
        "justificativa": record["justificativa"],
    }

cases = [format_case(r) for r in load_json(vector_db_content_filepath)]
for case in cases:
    resumo_clinico = case.pop("resumo_clinico")
    emb_resumo_clinico = await gerar_embedding(client, resumo_clinico)
    doc = Documento(
        caso_id=case.pop("caso_id"),
        resumo_clinico=resumo_clinico,
        metadados=case,
        embedding=emb_resumo_clinico,
    )
    store.adicionar(doc)

In [5]:
cases[0]

{'dut_id': 3,
 'dut_text': '# DUT 3 — ANGIOTOMOGRAFIA CORONARIANA\n\n> **Cobertura obrigatória** quando preenchido pelo menos um dos critérios\n> abaixo. Aplicável apenas em aparelhos **multislice com 64 colunas de\n> detectores ou mais**.\n\n---\n\n## Critério (a)\n\nAvaliação inicial de pacientes sintomáticos com probabilidade pré-teste\nde **10% a 70%**, calculada segundo os critérios de **Diamond-Forrester\nrevisado** (Genders et al., 2011), como opção aos outros métodos\ndiagnósticos de doença arterial coronariana.\n\n### Tabela de Probabilidade Pré-Teste em Pacientes com Dor Torácica (%)\n\n*Adaptado de T.S.S. Genders et al., 2011*\n\n| Idade | Angina Típica H | Angina Típica M | Angina Atípica H | Angina Atípica M | Dor não Anginosa H | Dor não Anginosa M |\n|-------|----------------:|----------------:|-----------------:|-----------------:|-------------------:|-------------------:|\n| 30–39 | 59,1 | 22,5 | 28,9 |  9,6 | 17,7 |  5,3 |\n| 40–49 | 68,9 | 36,7 | 38,4 | 14,0 | 24,8 |

In [7]:
1

1